# Customer Lifetime Value Estimation with BG/NBD and Gamma-Gamma Models

This notebook estimates customer lifetime value from retail transaction data using a two-stage probabilistic modelling framework. The BG/NBD model is used to estimate future purchase frequency, while the Gamma-Gamma model is used to estimate the expected monetary value of future transactions.

The workflow follows four main stages: data preparation, purchase-frequency modelling, monetary-value modelling, and customer segmentation. The modelling specification is kept consistent with the project report so that the numerical outputs remain comparable with the documented results.


## 1. Data Loading and Initial Inspection

The raw transaction dataset is loaded and inspected before modelling. This step checks the basic structure, available columns, and data types.


In [ ]:
import pandas as pd
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import gaussian_kde
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes.plotting import plot_probability_alive_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import silhouette_score


In [ ]:
retail_df = pd.read_csv("data/raw/data.csv")
retail_df.head()


In [ ]:
retail_df.info()


## 2. Data Preparation

The data is cleaned by converting the transaction date to datetime format, removing invalid transactions, dropping missing customer identifiers, and eliminating duplicate records. The final six months of the dataset are reserved as the holdout period for later comparison with model predictions.


In [ ]:
retail_df['Date'] = pd.to_datetime(retail_df['Date'])retail_clean = retail_df[    (retail_df['Total_Cost'] > 0) &    (retail_df['Total_Items'] > 0)].copy()retail_clean.dropna(subset=['Customer_Name'], inplace=True)retail_clean['Customer_Name'] = retail_clean['Customer_Name'].astype(str).str.strip()retail_clean.drop_duplicates(inplace=True)retail_clean = retail_clean.sort_values(['Customer_Name', 'Date']).reset_index(drop=True)print(retail_clean.shape)

### Exploratory Data Analysis

#### Dataset Overview

In [ ]:
basic_data_overview = pd.DataFrame({
    'Metric': [
        'Number of rows',
        'Number of columns',
        'Number of unique customers',
        'Start date',
        'End date',
        'Number of unique products',
        'Number of duplicate rows'
    ],
    'Value': [
        len(retail_clean),
        retail_clean.shape[1],
        retail_clean['Customer_Name'].nunique(),
        retail_clean['Date'].min(),
        retail_clean['Date'].max(),
        retail_clean['Product'].nunique() if 'Product' in retail_clean.columns else 'N/A',
        retail_clean.duplicated().sum()
    ]
})

basic_data_overview

#### Missing Values and Invalid Transaction Checks

In [ ]:
key_cols = ['Customer_Name', 'Date', 'Total_Cost', 'Total_Items']

available_key_cols = [col for col in key_cols if col in retail_clean.columns]

missing_check = (
    retail_clean[available_key_cols]
    .isna()
    .sum()
    .reset_index()
)

missing_check.columns = ['Column', 'Missing values']

invalid_transaction_check = pd.DataFrame({
    'Check': [
        'Transactions with Total_Cost <= 0',
        'Transactions with Total_Items <= 0'
    ],
    'Count': [
        (retail_clean['Total_Cost'] <= 0).sum(),
        (retail_clean['Total_Items'] <= 0).sum()
    ]
})

display(missing_check)
display(invalid_transaction_check)

In [ ]:
transaction_value_summary = retail_clean['Total_Cost'].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).to_frame(name='Total_Cost')

transaction_value_summary

#### Transaction Value Distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    retail_clean['Total_Cost'],
    bins=50
)

plt.xlabel('Total_Cost')
plt.ylabel('Number of transactions')
plt.title('Distribution of Transaction Values')
plt.tight_layout()
plt.show()

#### Transaction Trend Over Time

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    np.log1p(retail_clean['Total_Cost']),
    bins=50
)

plt.xlabel('log(1 + Total_Cost)')
plt.ylabel('Number of transactions')
plt.title('Log Distribution of Transaction Values')
plt.tight_layout()
plt.show()

In [ ]:
monthly_transactions = (
    retail_clean
    .set_index('Date')
    .resample('ME')
    .size()
    .reset_index(name='transaction_count')
)

monthly_transactions.head()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    monthly_transactions['Date'],
    monthly_transactions['transaction_count'],
    marker='o'
)

plt.xlabel('Month')
plt.ylabel('Number of transactions')
plt.title('Monthly Transaction Trend')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
calibration_end_date = retail_clean['Date'].max() - pd.DateOffset(months=6)

calibration_data = retail_clean[retail_clean['Date'] <= calibration_end_date].copy()
holdout_data = retail_clean[retail_clean['Date'] > calibration_end_date].copy()

holdout_days = (
    holdout_data['Date'].max().floor('D') -
    holdout_data['Date'].min().floor('D')
).days + 1

print("Calibration end date:", calibration_end_date)
print("Holdout start date:", holdout_data['Date'].min())
print("Holdout end date:", holdout_data['Date'].max())
print("Holdout length in days:", holdout_days)

- `frequency`: number of repeat purchase periods during the calibration period. Because the model uses `freq='D'`, this is interpreted as repeat purchase days/periods rather than raw transaction rows.
- `recency`: time of the most recent purchase.

In [ ]:
current_date = calibration_data['Date'].max()

rfm_summary = summary_data_from_transaction_data(
    calibration_data,
    customer_id_col='Customer_Name',
    datetime_col='Date',
    monetary_value_col='Total_Cost',
    observation_period_end=current_date,
    freq='D'
)

# BG/NBD + Gamma-Gamma model is fitted on repeat customers only
rfm_summary = rfm_summary[
    (rfm_summary['frequency'] > 0) &
    (rfm_summary['monetary_value'] > 0)
].copy()

rfm_summary.head()

### Additional Data Check: Impact of Filtering Repeat Customers

The Gamma-Gamma model requires customers with repeat purchases and positive monetary value. Therefore, the main modeling dataset keeps customers with `frequency > 0`.

This check measures how many customers are excluded because they are one-time buyers.

The purpose is to acknowledge that the final CLV model focuses on repeat customers, while one-time buyers would require a separate treatment or model in a real business setting.

In [ ]:
# Recreate RFMT summary before filtering frequency > 0
rfm_all_customers = summary_data_from_transaction_data(
    calibration_data,
    customer_id_col='Customer_Name',
    datetime_col='Date',
    monetary_value_col='Total_Cost',
    observation_period_end=calibration_data['Date'].max(),
    freq='D'
)

total_customers_before_filter = len(rfm_all_customers)
repeat_customers = (rfm_all_customers['frequency'] > 0).sum()
one_time_customers = (rfm_all_customers['frequency'] == 0).sum()

repeat_customer_filter_audit = pd.DataFrame({
    'Metric': [
        'Total customers before frequency filter',
        'Repeat customers kept (frequency > 0)',
        'One-time buyers excluded (frequency = 0)',
        'Share of customers kept',
        'Share of customers excluded'
    ],
    'Value': [
        total_customers_before_filter,
        repeat_customers,
        one_time_customers,
        repeat_customers / total_customers_before_filter,
        one_time_customers / total_customers_before_filter
    ]
})

repeat_customer_filter_audit

In [ ]:
rfmt_summary_stats = rfm_summary[
    ['frequency', 'recency', 'T', 'monetary_value']
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

rfmt_summary_stats

In [ ]:
def add_holdout_actuals(rfm_df, holdout_df):
    """
    Add holdout actuals aligned with Lifetimes daily frequency.

    actual_purchase_days_6m:
        Number of unique customer-day purchase periods in holdout.
        This aligns with freq='D' in Lifetimes.

    actual_transactions_6m:
        Raw number of transaction rows in holdout.
        Used only for monetary-value diagnostics.

    actual_value_6m:
        Total transaction value in holdout.
    """
    holdout_tmp = holdout_df.copy()
    holdout_tmp['Purchase_Day'] = holdout_tmp['Date'].dt.floor('D')

    actual_purchase_days = (
        holdout_tmp
        .drop_duplicates(['Customer_Name', 'Purchase_Day'])
        .groupby('Customer_Name')
        .size()
    )

    actual_transactions = holdout_tmp.groupby('Customer_Name').size()
    actual_value = holdout_tmp.groupby('Customer_Name')['Total_Cost'].sum()
    actual_avg_transaction_value = holdout_tmp.groupby('Customer_Name')['Total_Cost'].mean()

    rfm_df['actual_6m'] = actual_purchase_days.reindex(rfm_df.index).fillna(0)
    rfm_df['actual_transactions_6m'] = actual_transactions.reindex(rfm_df.index).fillna(0)
    rfm_df['actual_value_6m'] = actual_value.reindex(rfm_df.index).fillna(0)
    rfm_df['actual_avg_transaction_value_6m'] = (
        actual_avg_transaction_value.reindex(rfm_df.index)
    )

    rfm_df['actual_active_6m'] = (rfm_df['actual_6m'] > 0).astype(int)

    return rfm_df

rfm_summary = add_holdout_actuals(rfm_summary, holdout_data)

rfm_summary[
    ['pred_6m', 'actual_6m', 'actual_transactions_6m', 'actual_active_6m', 'actual_value_6m']
].head()

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T']
)

rfm_summary['pred_1m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    30,
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T']
)

rfm_summary['pred_6m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    holdout_days,
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T']
)

rfm_summary['pred_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    365,
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T']
)

rfm_summary['p_alive'] = bgf.conditional_probability_alive(
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T']
)

rfm_summary.head()


In [ ]:
rfm_summary = add_holdout_actuals(rfm_summary, holdout_data)rfm_summary[    ['pred_6m', 'actual_6m', 'actual_transactions_6m', 'actual_active_6m', 'actual_value_6m']].head()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    np.log1p(rfm_summary['frequency']),
    bins=50
)

plt.xlabel('log(1 + Frequency)')
plt.ylabel('Number of customers')
plt.title('Distribution of Repeat Purchase Frequency')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    rfm_summary['monetary_value'],
    bins=50
)

plt.xlabel('Monetary value')
plt.ylabel('Number of customers')
plt.title('Distribution of Customer-Level Monetary Value')
plt.tight_layout()
plt.show()

## 4. BG/NBD Model for Purchase Frequency

The BG/NBD model estimates the expected number of future purchases and the probability that a customer is still active. Predictions are generated for one-month, six-month, and twelve-month horizons.


In [ ]:
summary_bgnbd = pd.DataFrame({
    'Coef': bgf.params_,
    'Se (coef)': bgf.standard_errors_,
    'Lower 95% bound': bgf.confidence_intervals_['lower 95% bound'],
    'Upper 95% bound': bgf.confidence_intervals_['upper 95% bound']
})

summary_bgnbd


## 5. Purchase-Frequency Validation

The predicted six-month purchase frequency is compared with observed purchasing activity in the holdout period. The comparison is descriptive and provides an initial check of how closely the model approximates repeated transactions.


In [ ]:
max_trans = 6actual_counts = rfm_summary['actual_6m'].value_counts().sort_index()pred_counts = rfm_summary['pred_6m'].round().value_counts().sort_index()x = np.arange(max_trans + 1)actual_plot = [actual_counts.get(i, 0) for i in x]pred_plot = [pred_counts.get(i, 0) for i in x]plt.figure(figsize=(10, 6))width = 0.35plt.bar(x - width / 2, actual_plot, width, label='Actual')plt.bar(x + width / 2, pred_plot, width, label='Model')plt.xlabel('Repeat purchase days / periods')plt.ylabel('Number of customers')plt.title('BG/NBD Prediction vs Actual Holdout Purchase Periods')plt.xticks(x)plt.legend()plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(rfm_summary['actual_6m'], rfm_summary['pred_6m'], alpha=0.5)

plt.xlabel('Actual Purchases (6m)')
plt.ylabel('Predicted Purchases (6m)')
plt.title('Predicted vs Actual Purchases')
plt.plot([0, 10], [0, 10])
plt.tight_layout()
plt.show()


This step evaluates how well the BG/NBD model predicts the number of repeat purchase periods in the 6-month holdout period.

Because the model is estimated with `freq='D'`, the actual holdout target is defined as the number of unique customer-day purchase periods, not the raw number of transaction rows.

In [ ]:
# Actual number of transactions in holdout period
# actual_6m = holdout_data.groupby('Customer_Name').size()
# rfm_summary['actual_6m'] = actual_6m.reindex(rfm_summary.index).fillna(0)

# Predicted number of purchases in 6 months already exists as pred_6m
y_true = rfm_summary['actual_6m']
y_pred = rfm_summary['pred_6m']

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
bias = (y_pred - y_true).mean()
corr = y_true.corr(y_pred)

purchase_validation_metrics = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'Bias mean(pred - actual)', 'Correlation'],
    'Value': [mae, rmse, bias, corr]
})

purchase_validation_metrics

### Defining Actual Customer Activity in Holdout

This step creates a binary indicator called `actual_active_6m`.

A customer is defined as active in the holdout period if they made at least one transaction during the 6-month holdout window.

This variable is used to check whether customers with higher predicted `p_alive` are also more likely to purchase again in the holdout period.

In [ ]:
# A customer is considered active in the holdout period
# if they have at least 1 transaction in the 6-month holdout period.
rfm_summary['actual_active_6m'] = (rfm_summary['actual_6m'] > 0).astype(int)

rfm_summary[['p_alive', 'actual_6m', 'actual_active_6m']].head()

### Grouping Customers by Predicted p_alive

This step ranks customers by their predicted probability of being alive and divides them into 10 deciles.

- Decile 1 contains customers with the lowest predicted `p_alive`.
- Decile 10 contains customers with the highest predicted `p_alive`.

The purpose is to evaluate whether higher predicted `p_alive` groups also show stronger actual activity in the holdout period.

In [ ]:
# Rank first to avoid problems when many customers have identical p_alive values
rfm_summary['p_alive_rank'] = rfm_summary['p_alive'].rank(method='first')

# Divide customers into 10 groups based on predicted p_alive
# Decile 1 = lowest p_alive
# Decile 10 = highest p_alive
rfm_summary['p_alive_decile'] = pd.qcut(
    rfm_summary['p_alive_rank'],
    q=10,
    labels=False
) + 1

rfm_summary[['p_alive', 'p_alive_decile', 'actual_active_6m']].head()

### Comparing Predicted p_alive with Actual Holdout Activity

This step compares predicted `p_alive` with actual customer behavior in the holdout period.

For each `p_alive` decile, we calculate:

- the number of customers,
- the average predicted `p_alive`,
- the actual active rate in the 6-month holdout period,
- the average number of actual holdout transactions.

If higher `p_alive` deciles have higher actual active rates or higher actual transactions, this supports the use of `p_alive` as an indicator of future customer activity.

In [ ]:
p_alive_validation = (
    rfm_summary
    .groupby('p_alive_decile')
    .agg(
        customers=('actual_active_6m', 'size'),
        min_p_alive=('p_alive', 'min'),
        max_p_alive=('p_alive', 'max'),
        avg_predicted_p_alive=('p_alive', 'mean'),
        actual_active_rate=('actual_active_6m', 'mean'),
        avg_actual_transactions=('actual_6m', 'mean')
    )
    .reset_index()
)

p_alive_validation

### Visualization: p_alive Validation by Decile

This chart visualizes the relationship between predicted `p_alive` and actual active rate.

The chart helps show whether customers with higher predicted activity probability also have higher observed activity in the holdout period.

The two lines should not be interpreted as perfectly identical measures, because `p_alive` represents the probability that a customer is still active, while actual active rate is based on whether the customer made at least one transaction in the 6-month holdout period.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    p_alive_validation['p_alive_decile'],
    p_alive_validation['avg_predicted_p_alive'],
    marker='o',
    label='Average predicted p_alive'
)

plt.plot(
    p_alive_validation['p_alive_decile'],
    p_alive_validation['actual_active_rate'],
    marker='o',
    label='Actual active rate in holdout'
)

plt.xlabel('p_alive decile')
plt.ylabel('Rate')
plt.title('p_alive Validation by Decile')
plt.legend()
plt.tight_layout()
plt.show()

### Low-vs-High p_alive Comparison

This step compares the lowest and highest `p_alive` deciles directly.

The comparison provides a simple interpretation of model usefulness:

- customers in the lowest `p_alive` group should have weaker actual activity,
- customers in the highest `p_alive` group should have stronger actual activity.

This result is useful for explaining why `p_alive` is meaningful for customer segmentation.

In [ ]:
lowest_decile = p_alive_validation['p_alive_decile'].min()
highest_decile = p_alive_validation['p_alive_decile'].max()

comparison_low_high = p_alive_validation[
    p_alive_validation['p_alive_decile'].isin([lowest_decile, highest_decile])
]

comparison_low_high

### Interpretation of p_alive Validation

The validation result supports the usefulness of `p_alive`.

Customers in the lowest `p_alive` decile have an actual active rate of around 17.1% and only about 0.20 actual transactions on average in the holdout period.

In contrast, customers in the highest `p_alive` decile have an actual active rate of around 76.9% and about 2.27 actual transactions on average.

This means that customers predicted to be more likely alive by the BG/NBD model are also more likely to purchase again in the holdout period.

However, `p_alive` should not be interpreted as the exact probability of making a purchase within six months. It represents the model-estimated probability that the customer is still active, while actual holdout activity is measured by whether the customer made at least one observed transaction.

### Additional Robustness Check: BG/NBD vs Naive Baseline

This check compares the BG/NBD purchase prediction with a simple naive baseline.

The naive baseline assumes that each customer's future purchase count is proportional to their historical repeat purchase rate:

`naive_predicted_purchases = historical_purchase_rate × holdout_days`

The purpose is to check whether BG/NBD provides better predictive performance than a simple historical-rate rule.

This helps justify the use of BG/NBD instead of relying only on a simple average-based prediction.

In [ ]:
# Naive baseline:
# historical repeat purchase-period rate = frequency / T
# future purchase periods = historical rate × holdout_days
rfm_summary['naive_pred_6m'] = (
    rfm_summary['frequency'] / rfm_summary['T'].replace(0, np.nan)
) * holdout_days

rfm_summary['naive_pred_6m'] = rfm_summary['naive_pred_6m'].fillna(0)

y_true = rfm_summary['actual_6m']
y_pred_bgnbd = rfm_summary['pred_6m']
y_pred_naive = rfm_summary['naive_pred_6m']

baseline_comparison = pd.DataFrame({
    'Model': ['BG/NBD', 'Naive historical-rate baseline'],
    'MAE': [
        mean_absolute_error(y_true, y_pred_bgnbd),
        mean_absolute_error(y_true, y_pred_naive)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_true, y_pred_bgnbd)),
        np.sqrt(mean_squared_error(y_true, y_pred_naive))
    ],
    'Bias mean(pred - actual)': [
        (y_pred_bgnbd - y_true).mean(),
        (y_pred_naive - y_true).mean()
    ],
    'Correlation with actual': [
        y_true.corr(y_pred_bgnbd),
        y_true.corr(y_pred_naive)
    ]
})

baseline_comparison

## 6. Customer Activity Probability

The probability-alive output indicates how likely each customer is to remain active according to the BG/NBD model. The distribution and retention-probability matrix help interpret how purchase frequency and recency affect customer activity.


In [ ]:
plt.figure(figsize=(8, 5))

counts, bins, _ = plt.hist(
    rfm_summary['p_alive'],
    bins=30,
    alpha=0.6
)

kde = gaussian_kde(rfm_summary['p_alive'])
x_vals = np.linspace(rfm_summary['p_alive'].min(), rfm_summary['p_alive'].max(), 200)
y_vals = kde(x_vals)
y_vals_scaled = y_vals * len(rfm_summary['p_alive']) * (bins[1] - bins[0])

plt.plot(x_vals, y_vals_scaled, linewidth=2)
plt.title('Distribution of Probability Alive', fontsize=13)
plt.xlabel('p_alive')
plt.ylabel('Number of Customers')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
plot_probability_alive_matrix(bgf)

plt.title('Thermodynamic Map: Probability of Customer Being Alive')
plt.xlabel('Customer Historical Frequency')
plt.ylabel('Customer Recency')
plt.tight_layout()
plt.show()


## 7. Gamma-Gamma Model for Monetary Value

The Gamma-Gamma model estimates the expected average transaction value for each customer. This estimate is then combined with the BG/NBD purchase-frequency model to calculate customer lifetime value.


In [ ]:
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    rfm_summary['frequency'],
    rfm_summary['monetary_value']
)

rfm_summary['expected_avg_sales'] = ggf.conditional_expected_average_profit(
    rfm_summary['frequency'],
    rfm_summary['monetary_value']
)

rfm_summary.head()


In [ ]:
summary_gammagamma = pd.DataFrame({
    'Coef': ggf.params_,
    'Se (coef)': ggf.standard_errors_,
    'Lower 95% bound': ggf.confidence_intervals_['lower 95% bound'],
    'Upper 95% bound': ggf.confidence_intervals_['upper 95% bound']
})

summary_gammagamma


### Additional Validation 3: Gamma-Gamma Assumption Check

The Gamma-Gamma model is used to estimate each customer's expected average transaction value.

A key assumption of this model is that purchase frequency and monetary value are relatively independent. In other words, customers who purchase more often should not necessarily have much higher or much lower average transaction values.

This step checks the relationship between `frequency` and `monetary_value` using Pearson and Spearman correlation.

- **Pearson correlation** checks the linear relationship.
- **Spearman correlation** checks the rank-based monotonic relationship.

If the correlations are not too high, the Gamma-Gamma assumption is more acceptable for this project. If the correlations are high, this should be discussed as a limitation.

In [ ]:
# Compute Pearson and Spearman correlations
pearson_corr = rfm_summary['frequency'].corr(
    rfm_summary['monetary_value'],
    method='pearson'
)

spearman_corr = rfm_summary['frequency'].corr(
    rfm_summary['monetary_value'],
    method='spearman'
)

gg_assumption_check = pd.DataFrame({
    'Correlation type': ['Pearson', 'Spearman'],
    'Correlation between frequency and monetary_value': [pearson_corr, spearman_corr]
})

gg_assumption_check

### Visualization: Frequency vs Monetary Value

This scatter plot visualizes the relationship between purchase frequency and average monetary value.

The purpose is to check whether customers with higher purchase frequency also tend to have much higher average transaction value.

A very strong relationship may weaken the Gamma-Gamma model assumption. A weak or moderate relationship is more acceptable.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    rfm_summary['frequency'],
    rfm_summary['monetary_value'],
    alpha=0.3
)

plt.xlabel('Frequency')
plt.ylabel('Monetary value')
plt.title('Frequency vs Monetary Value')
plt.tight_layout()
plt.show()

### Visualization: Log Frequency vs Monetary Value

Because purchase frequency is usually skewed, this plot uses `log(1 + frequency)` to make the relationship easier to observe.

This chart is used as an additional visual check for the Gamma-Gamma assumption.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    np.log1p(rfm_summary['frequency']),
    rfm_summary['monetary_value'],
    alpha=0.3
)

plt.xlabel('log(1 + Frequency)')
plt.ylabel('Monetary value')
plt.title('Log Frequency vs Monetary Value')
plt.tight_layout()
plt.show()

In [ ]:
monetary_validation_df = rfm_summary[    rfm_summary['actual_transactions_6m'] > 0].copy()y_true = monetary_validation_df['actual_avg_transaction_value_6m']y_pred = monetary_validation_df['expected_avg_sales']monetary_validation_df['absolute_percentage_error'] = (    np.abs(y_pred - y_true) / y_true) * 100monetary_validation_df['percentage_error'] = (    (y_pred - y_true) / y_true) * 100mae = mean_absolute_error(y_true, y_pred)rmse = np.sqrt(mean_squared_error(y_true, y_pred))mape = monetary_validation_df['absolute_percentage_error'].mean()median_ape = monetary_validation_df['absolute_percentage_error'].median()wape = (np.abs(y_pred - y_true).sum() / y_true.sum()) * 100mean_percentage_bias = monetary_validation_df['percentage_error'].mean()corr = y_true.corr(y_pred)monetary_percentage_error_metrics = pd.DataFrame({    'Metric': [        'MAE',        'RMSE',        'MAPE (%)',        'Median APE (%)',        'WAPE (%)',        'Mean percentage bias (%)',        'Correlation'    ],    'Value': [        mae,        rmse,        mape,        median_ape,        wape,        mean_percentage_bias,        corr    ]})monetary_percentage_error_metrics

## 8. Customer Lifetime Value Estimation

Customer lifetime value is estimated by combining expected purchase frequency from the BG/NBD model with expected average transaction value from the Gamma-Gamma model.


In [ ]:
rfm_summary['CLV_6m'] = ggf.customer_lifetime_value(
    bgf,
    rfm_summary['frequency'],
    rfm_summary['recency'],
    rfm_summary['T'],
    rfm_summary['monetary_value'],
    time=6,
    discount_rate=0.01,
    freq='D'
)

print("Top 20 customers have the highest predicted CLV in the next 6 months:")
display(rfm_summary.sort_values(by='CLV_6m', ascending=False).head(20))


## 9. Customer Segmentation

K-Means clustering is applied using two model-based features: probability alive and predicted CLV. The resulting clusters are mapped into four business-oriented customer segments: Lost, At Risk, Potential, and VIP.


In [ ]:
cluster_features = rfm_summary[['p_alive', 'CLV_6m']]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_summary['KMeans_Cluster'] = kmeans.fit_predict(scaled_features)

cluster_profile = rfm_summary.groupby('KMeans_Cluster')[['p_alive', 'CLV_6m']].mean()
cluster_profile['Composite_Score'] = (
    cluster_profile['p_alive'].rank(method='first') +
    cluster_profile['CLV_6m'].rank(method='first')
)

ordered_clusters = cluster_profile.sort_values(['p_alive', 'CLV_6m']).index.tolist()
segment_labels = ['Lost', 'At Risk', 'Potential', 'VIP']
cluster_name_map = {cluster_id: segment_labels[idx] for idx, cluster_id in enumerate(ordered_clusters)}

rfm_summary['KMeans_Segment'] = rfm_summary['KMeans_Cluster'].map(cluster_name_map)
cluster_profile['KMeans_Segment'] = cluster_profile.index.map(cluster_name_map)
cluster_profile['Sample size'] = rfm_summary['KMeans_Cluster'].value_counts().reindex(cluster_profile.index).values
cluster_profile['Percentage of Customers'] = (
    rfm_summary['KMeans_Cluster']
    .value_counts(normalize=True)
    .reindex(cluster_profile.index)
    .values * 100
).round(2).astype(str)

cluster_profile = cluster_profile[
    ['p_alive', 'CLV_6m', 'KMeans_Segment', 'Composite_Score', 'Sample size', 'Percentage of Customers']
]

cluster_profile


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=rfm_summary,
    x='p_alive',
    y='CLV_6m',
    hue='KMeans_Segment',
    palette={'VIP': 'gold', 'At Risk': 'red', 'Potential': 'cyan', 'Lost': 'grey'},
    s=70
)

plt.title('K-Means Customer Segmentation by p_alive and CLV_6m')
plt.legend(title='KMeans_Segment')
plt.tight_layout()
plt.show()


### Additional Validation 4: Segment Holdout Validation

After creating the K-means customer segments, this step checks whether the segments are meaningful when compared with actual customer behavior in the 6-month holdout period.

The goal is to evaluate whether groups such as VIP, Potential, At-risk, and Lost show different actual active rates, transaction counts, and transaction values in the holdout data.

This validation helps assess whether the CLV-based segmentation is economically useful, not just technically generated by K-means.

In [ ]:
# Reuse holdout actuals created earlier.# actual_6m = unique customer-day purchase periods# actual_transactions_6m = raw transaction count# actual_value_6m = total holdout transaction valuerfm_summary = add_holdout_actuals(rfm_summary, holdout_data)rfm_summary[    ['actual_6m', 'actual_transactions_6m', 'actual_active_6m', 'actual_value_6m']].head()

In [ ]:
segment_col = 'KMeans_Segment'

segment_holdout_validation = (
    rfm_summary
    .groupby(segment_col)
    .agg(
        customers=(segment_col, 'size'),
        avg_predicted_CLV=('CLV_6m', 'mean'),
        avg_p_alive=('p_alive', 'mean'),
        actual_active_rate=('actual_active_6m', 'mean'),
        avg_actual_transactions_6m=('actual_6m', 'mean'),
        avg_actual_value_6m=('actual_value_6m', 'mean'),
        total_actual_value_6m=('actual_value_6m', 'sum')
    )
    .sort_values('avg_predicted_CLV', ascending=False)
)

segment_holdout_validation

### Visualization: Average actual holdout value by segment

In [ ]:
plot_df = segment_holdout_validation.copy()

plt.figure(figsize=(8, 5))

plt.bar(
    plot_df.index.astype(str),
    plot_df['avg_actual_value_6m']
)

plt.xlabel('Segment')
plt.ylabel('Average actual value in holdout')
plt.title('Average Actual Holdout Value by Segment')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Visualization: Actual active rate by segment

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    plot_df.index.astype(str),
    plot_df['actual_active_rate']
)

plt.xlabel('Segment')
plt.ylabel('Actual active rate in holdout')
plt.title('Actual Active Rate by Segment')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Visualization: Average actual transactions by segment

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    plot_df.index.astype(str),
    plot_df['avg_actual_transactions_6m']
)

plt.xlabel('Segment')
plt.ylabel('Average actual transactions in holdout')
plt.title('Average Actual Transactions by Segment')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Compare predicted CLV rank with actual holdout value rank

In [ ]:
segment_ranking_check = segment_holdout_validation.copy()

segment_ranking_check['predicted_CLV_rank'] = (
    segment_ranking_check['avg_predicted_CLV']
    .rank(ascending=False, method='dense')
)

segment_ranking_check['actual_value_rank'] = (
    segment_ranking_check['avg_actual_value_6m']
    .rank(ascending=False, method='dense')
)

segment_ranking_check[
    [
        'customers',
        'avg_predicted_CLV',
        'avg_actual_value_6m',
        'predicted_CLV_rank',
        'actual_value_rank',
        'avg_p_alive',
        'actual_active_rate',
        'avg_actual_transactions_6m'
    ]
]

### Additional Sensitivity Check: Revenue-Based CLV vs Assumed Profit-Based CLV

The CLV calculated in this project is revenue-based or transaction-value-based because the dataset does not include gross margin, campaign cost, service cost, or cost-to-serve.

This sensitivity check applies assumed margin rates to convert revenue-based CLV into approximate profit-based CLV.

The purpose is not to create a final profit model, but to show how the framework could be extended if margin data were available.

In [ ]:
margin_rates = [0.10, 0.20, 0.30, 0.40]

profit_sensitivity_results = []

for margin in margin_rates:
    temp = rfm_summary.copy()
    temp['assumed_profit_CLV_6m'] = temp['CLV_6m'] * margin
    
    segment_profit = (
        temp
        .groupby(segment_col)
        .agg(
            customers=(segment_col, 'size'),
            avg_revenue_based_CLV=('CLV_6m', 'mean'),
            avg_assumed_profit_CLV=('assumed_profit_CLV_6m', 'mean'),
            total_assumed_profit_CLV=('assumed_profit_CLV_6m', 'sum')
        )
        .reset_index()
    )
    
    segment_profit['assumed_margin_rate'] = margin
    
    profit_sensitivity_results.append(segment_profit)

profit_sensitivity = pd.concat(profit_sensitivity_results, ignore_index=True)

profit_sensitivity.sort_values(
    ['assumed_margin_rate', 'avg_assumed_profit_CLV'],
    ascending=[True, False]
)

### Additional Validation 5: K = 4 Robustness Check

The main analysis uses K = 4 in K-means clustering to create four CRM-actionable customer segments: Lost, At-risk, Potential, and VIP.

This step checks whether K = 4 is reasonable from a technical perspective by comparing different numbers of clusters using:

- **Elbow method**: evaluates how inertia decreases as the number of clusters increases.
- **Silhouette score**: evaluates how well-separated the clusters are.

This validation is used as a robustness check. The final choice of K should consider both technical metrics and business interpretability.

In [ ]:
# Use the same clustering features as the main K-means model
cluster_features_check = rfm_summary[['p_alive', 'CLV_6m']].copy()

# Standardize features because K-means is distance-based
scaler_check = StandardScaler()
scaled_features_check = scaler_check.fit_transform(cluster_features_check)

print("Clustering features prepared successfully.")
print("Shape of scaled features:", scaled_features_check.shape)

In [ ]:
k_range = range(2, 9)

inertias = []
silhouette_scores = []

# Use a sample for silhouette score to reduce computation time
sample_size = min(10000, scaled_features_check.shape[0])

rng = np.random.default_rng(42)
sample_idx = rng.choice(
    scaled_features_check.shape[0],
    size=sample_size,
    replace=False
)

scaled_sample = scaled_features_check[sample_idx]

for k in k_range:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels_full = km.fit_predict(scaled_features_check)
    
    # Inertia on full data
    inertias.append(km.inertia_)
    
    # Silhouette score on sampled data
    labels_sample = labels_full[sample_idx]
    sil_score = silhouette_score(scaled_sample, labels_sample)
    silhouette_scores.append(sil_score)

k_selection_result = pd.DataFrame({
    'K': list(k_range),
    'Inertia': inertias,
    'Silhouette_sample': silhouette_scores
})

k_selection_result

### Visualization: Elbow Method

The elbow method shows how inertia decreases as the number of clusters increases.

Inertia usually decreases when K increases, but after a certain point, the improvement becomes smaller. A reasonable K is often located around the point where the curve starts to flatten.

This chart is used as a technical reference for evaluating whether K = 4 is reasonable.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    k_selection_result['K'],
    k_selection_result['Inertia'],
    marker='o'
)

plt.xlabel('Number of clusters K')
plt.ylabel('Inertia')
plt.title('Elbow Method for K-means')
plt.tight_layout()
plt.show()

### Visualization: Silhouette Score by K

The silhouette score measures how well-separated the clusters are.

A higher silhouette score suggests clearer separation between clusters. However, the highest silhouette score is not always the best final choice, because the customer segments also need to be interpretable and useful for CRM strategy.

Therefore, this chart should be interpreted together with business logic.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    k_selection_result['K'],
    k_selection_result['Silhouette_sample'],
    marker='o'
)

plt.xlabel('Number of clusters K')
plt.ylabel('Silhouette score')
plt.title('Silhouette Score by K')
plt.tight_layout()
plt.show()

### Additional Check: Comparing K = 3, K = 4, and K = 5

Since the silhouette score is highest at K = 3, this step briefly compares K = 3, K = 4, and K = 5.

The purpose is not to replace the main K = 4 segmentation, but to check whether K = 4 remains reasonable when compared with nearby alternatives.

For each K, we examine the cluster size, average `p_alive`, and average six-month CLV to assess whether the resulting clusters are interpretable for CRM strategy.

In [ ]:
profile_results = []

for k in [3, 4, 5]:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = km.fit_predict(scaled_features_check)
    
    temp = rfm_summary[['p_alive', 'CLV_6m']].copy()
    temp['cluster'] = labels
    
    profile = (
        temp
        .groupby('cluster')
        .agg(
            customers=('cluster', 'size'),
            avg_p_alive=('p_alive', 'mean'),
            avg_CLV_6m=('CLV_6m', 'mean')
        )
        .reset_index()
    )
    
    profile['K'] = k
    profile['share'] = profile['customers'] / len(temp)
    
    profile_results.append(profile)

cluster_profile_comparison = pd.concat(profile_results, ignore_index=True)

cluster_profile_comparison = cluster_profile_comparison[
    ['K', 'cluster', 'customers', 'share', 'avg_p_alive', 'avg_CLV_6m']
].sort_values(['K', 'avg_CLV_6m'], ascending=[True, False])

cluster_profile_comparison